# EDA 01 — Profiling sistemático de los 4 CSVs

**Plan REQ-001 · Fase 1.1 · Owner: data-analyst + rugpull-domain-expert**

Objetivos:
- Schema, tipos, NaN%, cardinalidad de cada CSV.
- Rango temporal (block_min/block_max) y cobertura.
- Distribución de `is_rugpull` y balance de clases.
- Cruce pools ↔ tokens ↔ metadata ↔ events (integridad referencial).
- Identificar columnas candidatas a features (P0) para la Fase 2.1.

Datasets (en `api_datos/data/`):
| Archivo | Tamaño | Rol |
|---|---|---|
| `pool_list_complete.csv` | 412 KB | Catálogo de pools Uniswap V2 con `is_rugpull` |
| `token_metadata_complete.csv` | 178 KB | Metadatos de tokens (name, symbol, decimals, supply) |
| `eventos_pool_sync_mint_burn.csv` | 512 MB | Eventos SYNC/MINT/BURN por pool |
| `eventos_transfers_tokens.csv` | 2.1 GB | Transfers ERC-20 de tokens |

Herramienta: **DuckDB** (SQL sobre CSV sin cargar a RAM).

## 0. Setup

In [ ]:
import os
from pathlib import Path
import duckdb
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)
pd.set_option('display.float_format', '{:,.4f}'.format)

# Ruta al dataset. Desde el container jupyter, api_datos/data se monta — ajustar según compose.
# Para ejecución local fuera del container:
DATA_DIR = Path(os.getenv('DATA_DIR', '../../../api_datos/data')).resolve()
assert DATA_DIR.exists(), f'DATA_DIR no existe: {DATA_DIR}'

CSVS = {
    'pools':     DATA_DIR / 'pool_list_complete.csv',
    'metadata':  DATA_DIR / 'token_metadata_complete.csv',
    'events':    DATA_DIR / 'eventos_pool_sync_mint_burn.csv',
    'transfers': DATA_DIR / 'eventos_transfers_tokens.csv',
}
for name, path in CSVS.items():
    assert path.exists(), f'Falta {name}: {path}'
    print(f'{name:12s} {path.stat().st_size / 1024**2:>8.1f} MB  {path}')

con = duckdb.connect(':memory:')
con.execute("PRAGMA threads=4")  # ajustar según cores disponibles
print('DuckDB ready.')

## 1. Schema, tipos y row counts

In [ ]:
def describe_csv(name: str, path: Path) -> pd.DataFrame:
    """DESCRIBE + row count sin materializar."""
    schema = con.execute(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
    n = con.execute(f"SELECT count(*) AS n FROM read_csv_auto('{path}')").fetchone()[0]
    print(f'── {name} — {n:,} filas — {len(schema)} columnas')
    return schema

schemas = {name: describe_csv(name, path) for name, path in CSVS.items()}

In [ ]:
for name, df in schemas.items():
    print(f'\n### {name}')
    print(df[['column_name', 'column_type', 'null']].to_string(index=False))

## 2. NaN% y cardinalidad por columna

In [ ]:
def null_and_cardinality(name: str, path: Path) -> pd.DataFrame:
    schema = schemas[name]
    cols = schema['column_name'].tolist()
    n_total = con.execute(f"SELECT count(*) FROM read_csv_auto('{path}')").fetchone()[0]
    rows = []
    for col in cols:
        q = f"""
        SELECT
            {n_total} - count("{col}") AS n_null,
            count(DISTINCT "{col}") AS n_distinct
        FROM read_csv_auto('{path}')
        """
        n_null, n_distinct = con.execute(q).fetchone()
        rows.append({
            'column': col,
            'null_pct': round(100 * n_null / n_total, 2) if n_total else 0,
            'distinct': n_distinct,
            'cardinality_pct': round(100 * n_distinct / n_total, 2) if n_total else 0,
        })
    return pd.DataFrame(rows)

for name, path in CSVS.items():
    if name == 'transfers':
        print(f'\n── {name}: skip cardinality per-column (2 GB — demasiado lento en dev). Ver celda dedicada.')
        continue
    print(f'\n### {name}')
    print(null_and_cardinality(name, path).to_string(index=False))

## 3. Pools — distribución de `is_rugpull`

Clave para validar el target. Según README, el dataset incluye pools rug pulled y pools normales.

In [ ]:
# Balance de clases
df = con.execute(f"""
    SELECT is_rugpull, count(*) AS n
    FROM read_csv_auto('{CSVS['pools']}')
    GROUP BY 1 ORDER BY 1
""").df()
df['pct'] = (100 * df['n'] / df['n'].sum()).round(2)
print(df)

# Columnas clave del catálogo
sample = con.execute(f"SELECT * FROM read_csv_auto('{CSVS['pools']}') LIMIT 5").df()
print('\nMuestra:')
print(sample)

## 4. Metadata — decimals, total_supply

In [ ]:
sample = con.execute(f"SELECT * FROM read_csv_auto('{CSVS['metadata']}') LIMIT 5").df()
print('Muestra metadata:')
print(sample)

# Distribución de decimals
print('\n── Distribución decimals:')
print(con.execute(f"""
    SELECT decimals, count(*) AS n
    FROM read_csv_auto('{CSVS['metadata']}')
    GROUP BY 1 ORDER BY 2 DESC
""").df())

## 5. Events — tipos, rango de bloques, distribución

In [ ]:
print('── Tipos de evento:')
print(con.execute(f"""
    SELECT event_type, count(*) AS n
    FROM read_csv_auto('{CSVS['events']}')
    GROUP BY 1 ORDER BY 2 DESC
""").df())

print('\n── Rango de bloques:')
print(con.execute(f"""
    SELECT
        min(block_number) AS block_min,
        max(block_number) AS block_max,
        count(DISTINCT pair_address) AS n_pairs
    FROM read_csv_auto('{CSVS['events']}')
""").df())

## 6. Transfers — volumen y rango (operación costosa, 2 GB)

In [ ]:
print('── Transfers (puede tardar 30-60 s):')
print(con.execute(f"""
    SELECT
        count(*) AS n_transfers,
        min(block_number) AS block_min,
        max(block_number) AS block_max,
        count(DISTINCT token_address) AS n_tokens
    FROM read_csv_auto('{CSVS['transfers']}')
""").df())

## 7. Integridad referencial — pools ↔ metadata ↔ events ↔ transfers

In [ ]:
# Pools sin metadata
print('── Tokens (non-wETH) en pools SIN metadata:')
print(con.execute(f"""
    WITH pools AS (SELECT * FROM read_csv_auto('{CSVS['pools']}')),
         meta  AS (SELECT * FROM read_csv_auto('{CSVS['metadata']}'))
    SELECT count(*) AS pools_sin_meta
    FROM pools p
    LEFT JOIN meta m ON lower(p.token_address) = lower(m.token_address)
    WHERE m.token_address IS NULL
""").df())

# Pairs en events que NO están en pools
print('\n── Pairs en events NO presentes en pool_list:')
print(con.execute(f"""
    WITH ev AS (SELECT DISTINCT pair_address FROM read_csv_auto('{CSVS['events']}')),
         po AS (SELECT DISTINCT pair_address FROM read_csv_auto('{CSVS['pools']}'))
    SELECT count(*) AS pairs_huerfanos
    FROM ev LEFT JOIN po USING (pair_address)
    WHERE po.pair_address IS NULL
""").df())

## 8. Hallazgos y decisiones (completar al ejecutar)

| # | Hallazgo | Impacto | Decisión |
|---|----------|---------|----------|
| 1 | {balance is_rugpull} | {¿class imbalance?} | {SMOTE/class_weight/sampling} |
| 2 | {NaN% crítico en X} | {feature inviable} | {imputar/excluir} |
| 3 | {pools sin metadata} | {cobertura del target} | {filtrar/flag} |
| 4 | {pairs huérfanos} | {integridad DAG Airflow} | {validar en data contract} |
| 5 | {rango temporal real} | {alinea con NUM_BATCHES=12} | {confirmar meta jun-2020→may-2021} |

Registrar hallazgos en `.claude/memory/helix-bitacora.md` (sección Decisiones de Diseño).

**Próximo notebook:** `eda_02_batches.ipynb` — perfilado por batch mensual (detección de batches anómalos).